### Main ###

In [20]:
### Main processing script for SPAD data ###

### HEADER ###
from pathlib import Path
import numpy as np
import tifffile as tiff
import tqdm

from pySPADutils import pySPADutils as util
from grouping import group_bp_fmn


path = Path("H:\\KH260421-2D-beads-and-rings\\beads100nm-ex647\\SPAD\\FOV_3")

files = [f"Z_{i}.bin" for i in range(50000, 50001, 100)]

flag_preview = 0
flag_grouping = 1


## TODO: 
# ROI cropping
# Save BP as TIFF


for f in files:
    print(f"Reading {f}")
    bytearrayFile = util.readBinBig(path / f)

    print(f"Unpacking {f}")
    img = util.unpackBytearray(bytearrayFile) # Shape (X, Y, Frames)
    img = np.flip(img, axis=1)  # Flip along vertical axis to match sCMOS image

    
    ## TODO: go through all ROI


    match flag_preview:
        case 1:
            print(f"Summing {f}")
            z_sum = np.sum(img, axis=2, dtype=np.uint16)
            
            print(f"Saving {f} as TIF")
            tiff.imwrite(
                str(path / f.replace('.bin', '_zsum.tif')),
                z_sum,
                photometric="minisblack",
                compression="zlib", 
            )
    ## ROI, Shape (X, Y, Frames)， X and Y flip in python
    ystart, xstart = 239,186
    width, height = 40,40
    img = img[xstart:xstart+width, ystart:ystart+height, :]

    ## TODO: 
    # change shape format 
    # grouping method, if there is a faster one
    match flag_grouping:
        case 1:
            img_f = np.transpose(img, (2, 0, 1)) # Shape -> (Frames, X, Y)
            data_grouped, idx = group_bp_fmn(img_f, mode=1, out_frames = 20, bp_per_frame=100000/20)
            
            data_grouped = np.transpose(data_grouped, (1, 2, 0))
            print(f"Saving {f} as TIF")
            util.writeTiffBig(path / f.replace('.bin', '_grouped1.tif'), data_grouped)

    ## TODO: 
    # Modify the write TiffBig function to save as 16-bit  
    # Save the Sum of the frames as a 16-bit TIFF for visualization

    ## TODO:
    # Clean image
    # write it into function

Reading Z_50000.bin
Unpacking Z_50000.bin
oo Start grouping
|| mode 1 Sequential even grouping
** Done
Saving Z_50000.bin as TIF


In [ ]:
from pathlib import Path
from pySPADutils import pySPADutils as util
import numpy as np
from grouping import group_bp_fmn


def process_spad_folders(
    paths,
    files=None,
    mode=4,
    out_frames=20,
    bp_per_frame=10000,
):
    """
    Process SPAD .bin files from multiple folders.

    Parameters
    ----------
    paths : list[str | Path]
        List of directories containing .bin files.
    files : list[str] | None
        Specific filenames to process in each folder.
        If None, process all .bin files in each folder.
    mode : int
        group_bp_fmn mode.
    out_frames : int
        Number of output frames.
    bp_per_frame : int
        Bit-planes per frame.
    """
    for folder in paths:
        folder = Path(folder)

        if not folder.exists():
            print(f"Skipping missing folder: {folder}")
            continue

        if files is None:
            file_list = sorted(folder.glob("*.bin"))
        else:
            file_list = [folder / f for f in files]

        for file_path in file_list:
            if not file_path.exists():
                print(f"Skipping missing file: {file_path}")
                continue

            print(f"\nProcessing: {file_path}")

            bytearrayFile = util.readBinBig(file_path)

            print(f"Unpacking {file_path.name}")
            img = util.unpackBytearray(bytearrayFile)
            img_f = np.transpose(img, (2, 0, 1))

            data_grouped, idx = group_bp_fmn(
                img_f,
                mode=mode,
                out_frames=out_frames,
                bp_per_frame=bp_per_frame
            )

            data_grouped = np.transpose(data_grouped, (1, 2, 0))

            out_path = file_path.with_name(file_path.stem + "_grouped.tif")
            print(f"Saving: {out_path}")
            util.writeTiffBig(out_path, data_grouped)


# Example usage:
paths = [
    "H:/pySPAD/KH260407_TOMM20/SPAD/FOV_5",
    "H:/pySPAD/KH260407_TOMM20/SPAD/FOV_6",
    "H:/pySPAD/KH260407_TOMM20/SPAD/FOV_7",
    "H:/pySPAD/KH260407_TOMM20/SPAD/FOV_8"
]

files = [f"Z_{i}.bin" for i in range(49000, 51001, 100)]

process_spad_folders(paths, files=files)

In [ ]:
### Testing different grouping method ###
from pathlib import Path
from pySPADutils import pySPADutils as util
import numpy as np



### START HERE ###
# Generate list of files to process
path = Path("H:/pySPAD/KH260407_TOMM20/SPAD/FOV_1")
files = [f"Z_{i}.bin" for i in range(49900, 50001, 100)]

# Process each file
for f in files:
    
    # Read binary file into bytearray
    print(f"Unpacking {f}")
    bytearrayFile = util.readBinBig(path / f)
    img = util.unpackBytearray(bytearrayFile)  # (Z, H, W) -> (Z, Y, X) in ImageJ convention
    img = np.flip(img, axis=2)  # Flip X-axis to match sCMOS image
    
    # Get all indices for random sampling
    print("Getting random indices for resampling")
    N = img.shape[0]  # Total number of frames
    n_trials = 20
    n_draws = 10000
    idx = np.random.randint(0, N, size=(n_trials, n_draws))

    # Resampling and Summing in one step
    print("Resampling and summing")
    result = img[idx].sum(axis=1)   # shape: (20, 512, 512)
    
    # Save the grouped data as TIF
    print(f"Saving {f} as TIFF")
    util.writeTiffBig(path / f.replace('.bin', '_grouped.tif'), result)   




### Example ###

In [ ]:
bytearrayFile = util.readBinBig("test.bin")
bytearrayFile = util.readBinFolder(r"C:\Users\storm\Desktop\SPAD_python\testdata\acq00000")
img = util.unpackBytearray(bytearrayFile)